# unbroadcast-pattern — faded example 1: Peel leading broadcast axes

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbroadcast-pattern`. The last cell reports your progress on the `Backprop: Unbroadcast pattern` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbroadcast pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbroadcast-pattern`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbroadcast-pattern"
DD_SUBTOPIC = "Backprop: Unbroadcast pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When `grad` has more dimensions than `original`, the extra axes were prepended by broadcasting and must be summed away. The loop sums `dim=0` repeatedly until the ranks match.

## Faded exercise 1

### Peel the prepended axes

Implement `peel_leading(grad, original)` returning `grad` with all leading broadcast axes summed out so its rank equals `original`'s. Complete the in-loop statement that removes one leading axis per iteration.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def peel_leading(grad: t.Tensor, original: t.Tensor) -> t.Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    return grad

print(peel_leading(t.ones(2, 3, 4), t.zeros(3, 4)).shape)


def _test():
    original = t.zeros(3, 4)
    grad = t.ones(2, 3, 4)
    out = peel_leading(grad, original)
    assert out.shape == (3, 4), out.shape
    # independent truth: summing axis 0 of all-ones(2,3,4) gives 2 everywhere
    assert t.allclose(out, t.full((3, 4), 2.0)), out
    # already-matching rank is a no-op
    g2 = t.randn(5, 6)
    assert t.equal(peel_leading(g2, t.zeros(5, 6)), g2)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def peel_leading(grad: t.Tensor, original: t.Tensor) -> t.Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    return grad

print(peel_leading(t.ones(2, 3, 4), t.zeros(3, 4)).shape)
```
</details>